In [ ]:
# =====================================================================
# Two-Phase Pareto Front: Surrogate NSGA-II + Physics Verification
#   Phase 1: NSGA-II with surrogate (fast, μs/eval)
#   Phase 2: Re-rank top candidates with physical grid integral
# =====================================================================

import torch
import torch.nn as nn
import numpy as np
import math
import matplotlib.pyplot as plt
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.termination import get_termination
from pymoo.optimize import minimize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ============================================================
# 1. Physical Grid + Channel (copy from grid_search/ pareto_front)
# ============================================================
room_x, room_y, room_z_max = 10.0, 10.0, 3.0
x_BS, y_BS, z_BS = 10.0, -100.0, -10.0
E=8; d_B=0.075; lambda_val=0.075; L1=2
d_ref=abs(y_BS)*1.5; P_BS_dBm=40.0; R_th=0.2
N0_dBm_Hz=-174.0; B=20e6; p_m=1.0/5.0; n_users=5
P_BS=10**(P_BS_dBm/10.0)*1e-3; N0=10**(N0_dBm_Hz/10.0)*1e-3*B

GRID_RES=200; Z_FIXED=1.5
x_vals=torch.linspace(0,room_x,GRID_RES,device=device)
y_vals=torch.linspace(0,room_y,GRID_RES,device=device)
Xg,Yg=torch.meshgrid(x_vals,y_vals,indexing='ij')
grid_locs=torch.stack([Xg.flatten(),Yg.flatten(),torch.full_like(Xg.flatten(),Z_FIXED)],dim=1)
N_GRID=grid_locs.shape[0]

hotspot_center=torch.tensor([room_x/2,room_y/2],device=device); sigma_h=2.5
dsq=(grid_locs[:,0]-hotspot_center[0])**2+(grid_locs[:,1]-hotspot_center[1])**2
fs=1.0/(2.0*math.pi*sigma_h**2)*torch.exp(-dsq/(2.0*sigma_h**2))
fu=torch.full_like(fs,1.0/(room_x*room_y))
grid_weights=(0.7*fs+0.3*fu); grid_weights=grid_weights/grid_weights.sum()

np.random.seed(42)
_nlos_psi=torch.tensor(2*np.pi*np.random.rand(N_GRID),dtype=torch.float32,device=device)
_nlos_tt=torch.tensor(-np.pi+2*np.pi*np.random.rand(N_GRID),dtype=torch.float32,device=device)
_nlos_eta=torch.tensor(10**((-15+5*np.random.rand(N_GRID))/10),dtype=torch.float32,device=device)

def physical_channel(window_params,locs):
    if not isinstance(window_params,torch.Tensor):
        window_params=torch.tensor(window_params,dtype=torch.float32,device=device)
    if window_params.dim()==1: window_params=window_params.unsqueeze(0)
    xc,zc,Lx,Lz=window_params[:,0],window_params[:,1],window_params[:,2],window_params[:,3]
    Bn=window_params.shape[0]; xu,yu,zu=locs[:,0],locs[:,1],locs[:,2]
    dx_BS=xc-x_BS; dy_BS=torch.full_like(xc,0.0-y_BS); dz_BS=zc-z_BS
    R_BW=torch.sqrt(dx_BS**2+dy_BS**2+dz_BS**2)
    theta_BW=torch.atan2(dy_BS,dx_BS); phi_BW=torch.acos(dz_BS/R_BW)
    k_tx=torch.sin(phi_BW)*torch.cos(theta_BW); k_tz=torch.cos(phi_BW)
    x1,z1=xc-Lx/2,zc-Lz/2; x2,z2=xc-Lx/2,zc+Lz/2; x3,z3=xc+Lx/2,zc-Lz/2; x4,z4=xc+Lx/2,zc+Lz/2
    def rd(xs,zs):
        dx=xs.unsqueeze(1)-x_BS; dy=torch.full_like(dx,0.0-y_BS); dz=zs.unsqueeze(1)-z_BS
        L=torch.sqrt(dx**2+dy**2+dz**2); return dx/L,dy/L,dz/L
    ux1,_,uz1=rd(x1,z1); ux2,_,uz2=rd(x2,z2); ux3,_,uz3=rd(x3,z3); ux4,_,uz4=rd(x4,z4)
    dx_WU=xu-x_BS; dy_WU=yu-y_BS; dz_WU=zu-z_BS
    L_USER=torch.sqrt(dx_WU**2+dy_WU**2+dz_WU**2)
    ux_user=dx_WU/L_USER; uz_user=dz_WU/L_USER
    ux_all=torch.stack([ux1,ux2,ux3,ux4],dim=0); uz_all=torch.stack([uz1,uz2,uz3,uz4],dim=0)
    ux_min=ux_all.min(dim=0).values; ux_max=ux_all.max(dim=0).values
    uz_min=uz_all.min(dim=0).values; uz_max=uz_all.max(dim=0).values
    beta=1000.0
    inx=torch.sigmoid(beta*(ux_user-ux_min))*torch.sigmoid(beta*(ux_max-ux_user))
    inz=torch.sigmoid(beta*(uz_user-uz_min))*torch.sigmoid(beta*(uz_max-uz_user))
    illum=inx*inz
    dx_WU2=xu-xc.unsqueeze(1); dy_WU2=yu; dz_WU2=zu-zc.unsqueeze(1)
    R_WU=torch.sqrt(dx_WU2**2+dy_WU2**2+dz_WU2**2)
    t1=dx_WU2/R_WU; t2=dz_WU2/R_WU
    ax=(1.0-illum)*(k_tx.unsqueeze(1)-t1); az=(1.0-illum)*(k_tz.unsqueeze(1)-t2)
    sincx=torch.sinc((math.pi/lambda_val)*Lx.unsqueeze(1)*ax)
    sincz=torch.sinc((math.pi/lambda_val)*Lz.unsqueeze(1)*az)
    n_ant=torch.arange(E,dtype=torch.float32,device=device)
    ph1=(2.0*math.pi/lambda_val)*d_B*n_ant*torch.sin(theta_BW).unsqueeze(1)
    a1=(1.0/math.sqrt(E))*torch.complex(torch.cos(ph1),torch.sin(ph1))
    v1m=(lambda_val/(4.0*math.pi))/R_BW; v1p=-(2.0*math.pi/lambda_val)*R_BW
    v1=torch.complex(v1m*torch.cos(v1p),v1m*torch.sin(v1p)); H1=v1.unsqueeze(1)*a1.conj()
    ph2=(2.0*math.pi/lambda_val)*d_B*n_ant.unsqueeze(0)*torch.sin(_nlos_tt).unsqueeze(1)
    a2=(1.0/math.sqrt(E))*torch.complex(torch.cos(ph2),torch.sin(ph2))
    v2m=_nlos_eta*(lambda_val/(4.0*math.pi*d_ref))
    v2=torch.complex(v2m*torch.cos(_nlos_psi),v2m*torch.sin(_nlos_psi)); H2=v2.unsqueeze(1)*a2.conj().unsqueeze(0)
    H_total=math.sqrt(E/L1)*(H1.unsqueeze(1)+H2)
    fm=(Lx.unsqueeze(1)*Lz.unsqueeze(1)*sincx*sincz)/(lambda_val*R_WU)
    fp=(-(2.0*math.pi/lambda_val)*(k_tx*xc+k_tz*zc)-(math.pi/2.0))
    factor=torch.complex(fm*torch.cos(fp.unsqueeze(1)),fm*torch.sin(fp.unsqueeze(1)))
    return H_total*factor.unsqueeze(2)

@torch.no_grad()
def physics_outage(X_np):
    theta=torch.tensor(X_np,dtype=torch.float32,device=device)
    H_eq=physical_channel(theta,grid_locs)
    H_w=torch.sum(H_eq,dim=2)/math.sqrt(E)
    dp=(torch.abs(H_w)**2)*p_m*P_BS; intf=(n_users-1)*dp
    sinr=dp/(intf+N0); rate=torch.log2(1.0+sinr)
    bits=(rate<R_th).float()
    return torch.sum(bits*grid_weights,dim=1).cpu().numpy()

print(f'Physical grid built: {N_GRID} points')

# ============================================================
# 2. Load Surrogate Model
# ============================================================
class ResidualMLP(nn.Module):
    def __init__(self,in_dim=4,hidden=256,n_layers=4,dropout=0.05):
        super().__init__()
        self.input_layer=nn.Sequential(nn.Linear(in_dim,hidden),nn.ReLU(),nn.BatchNorm1d(hidden))
        self.blocks=nn.ModuleList([nn.Sequential(nn.Linear(hidden,hidden),nn.ReLU(),nn.BatchNorm1d(hidden),nn.Dropout(dropout)) for _ in range(n_layers)])
        self.output_layer=nn.Linear(hidden,1)
    def forward(self,x):
        h=self.input_layer(x)
        for b in self.blocks: h=h+b(h)
        return torch.sigmoid(self.output_layer(h))

# Auto-detect model path (try Kaggle input → working dir)
import os, glob
model_path = None
for candidate in [
    'surrogate_model.pt',
    '/kaggle/working/surrogate_model.pt',
] + sorted(glob.glob('/kaggle/input/**/surrogate_model.pt', recursive=True)):
    if os.path.exists(candidate):
        model_path = candidate; break
if model_path is None:
    raise FileNotFoundError(
        'surrogate_model.pt not found. Upload it via Kaggle Input or to working dir.')
print(f'Loading model from: {model_path}')
ckpt=torch.load(model_path,weights_only=False,map_location=device)
lb=ckpt['lb'].cpu().numpy(); ub=ckpt['ub'].cpu().numpy()
smodel=ResidualMLP().to(device); smodel.load_state_dict(ckpt['model_state']); smodel.eval()
print(f'Surrogate loaded: R²={ckpt["r2"]:.4f}')

def surrogate_outage(X_np):
    X_norm=(X_np-lb)/(ub-lb)
    X_t=torch.tensor(X_norm,dtype=torch.float32,device=device)
    with torch.no_grad(): return smodel(X_t).cpu().numpy().flatten()

# ============================================================
# 3. Phase 1 — NSGA-II with Surrogate
# ============================================================
class SurrogateProblem(ElementwiseProblem):
    def __init__(self):
        super().__init__(n_var=4,n_obj=2,n_ieq_constr=4,xl=lb,xu=ub)
    def _evaluate(self,x,out,*args,**kwargs):
        xc,zc,Lx,Lz=x[0],x[1],x[2],x[3]
        out["G"]=[Lx/2-xc, xc+Lx/2-10.0, Lz/2-zc, zc+Lz/2-3.0]
        out["F"]=[Lx*Lz, float(surrogate_outage(x.reshape(1,-1))[0])]

problem=SurrogateProblem()
algorithm=NSGA2(pop_size=500,n_offsprings=250,
    sampling=FloatRandomSampling(),crossover=SBX(prob=0.9,eta=15),
    mutation=PM(prob=0.25,eta=20),eliminate_duplicates=True)
termination=get_termination("n_gen",300)

print("\nPhase 1: NSGA-II with surrogate (pop=500, gen=300)...")
res=minimize(problem,algorithm,termination,seed=42,verbose=True)
print(f'Surrogate Pareto: {len(res.F)} points')

# ============================================================
# 4. Phase 2 — Physics Verification on Top Candidates
# ============================================================
# Pick candidates with surrogate outage < 25% (wide net)
mask_surr = res.F[:,1] < 0.25
candidates_X = res.X[mask_surr]
candidates_F_surr = res.F[mask_surr]
print(f'\nPhase 2: Verifying {len(candidates_X)} candidates with physics...')

# Batch-evaluate with physical model (batch to avoid OOM)
BATCH=200
physics_out = []
for i in range(0,len(candidates_X),BATCH):
    batch=candidates_X[i:i+BATCH]
    physics_out.append(physics_outage(batch))
    if i%BATCH==0: print(f'  ...{i}/{len(candidates_X)}')
physics_out=np.concatenate(physics_out)

# Build physics-verified Pareto dataset
F_physics = np.column_stack([candidates_F_surr[:,0], physics_out])  # [area, true_outage]
X_physics = candidates_X

# ============================================================
# 5. Plot — Surrogate vs Physics Pareto Front
# ============================================================
fig,axes=plt.subplots(1,2,figsize=(14,5.5))

# Non-dominated filter for physics results
def nondom(F):
    dom=np.zeros(len(F),dtype=bool)
    for i in range(len(F)):
        for j in range(len(F)):
            if i!=j and F[j,0]<=F[i,0] and F[j,1]<=F[i,1] and (F[j,0]<F[i,0] or F[j,1]<F[i,1]):
                dom[i]=True; break
    pf=F[~dom]; idx=np.argsort(pf[:,1]); return pf[idx]

pf_surr=nondom(res.F)
pf_phys=nondom(F_physics)

# Left: surrogate (all + pareto)
ax=axes[0]
ax.scatter(res.F[:,1]*100,res.F[:,0],c='steelblue',s=2,alpha=0.12)
ax.plot(pf_surr[:,1]*100,pf_surr[:,0],'o-',color='orange',markersize=3,linewidth=1.2,label=f'Surrogate PF')
ax.axvline(x=10,color='red',ls='--',alpha=0.5); ax.set_xlim(0,25); ax.set_ylim(0,None)
ax.set_xlabel('Outage [%]'); ax.set_ylabel('Area [m²]')
ax.set_title('Phase 1: Surrogate NSGA-II'); ax.legend(); ax.grid(alpha=0.3)

# Right: physics-verified
ax=axes[1]
ax.scatter(F_physics[:,1]*100,F_physics[:,0],c='gray',s=3,alpha=0.3,label='Candidates (physics)')
ax.plot(pf_phys[:,1]*100,pf_phys[:,0],'o-',color='darkgreen',markersize=5,linewidth=1.5,label=f'Physics PF ({len(pf_phys)} pts)')
ax.plot(pf_surr[:,1]*100,pf_surr[:,0],'--',color='orange',linewidth=1,alpha=0.6,label='Surrogate PF (ref)')
ax.axvline(x=10,color='red',ls='--',alpha=0.5)

feas=pf_phys[:,1]<=0.10
if feas.any():
    bi=np.argmin(pf_phys[feas,0])
    ba,bo=pf_phys[feas,0][bi],pf_phys[feas,1][bi]
    ax.plot(bo*100,ba,'r*',markersize=16,markeredgewidth=2,label=f'Knee: {ba:.4f}m² @ {bo*100:.1f}%')
ax.set_xlabel('Outage [%]'); ax.set_ylabel('Area [m²]')
ax.set_title('Phase 2: Physics-Verified Pareto Front'); ax.legend(); ax.grid(alpha=0.3)
ax.set_xlim(0,25); ax.set_ylim(0,None)

plt.tight_layout(); plt.savefig('two_phase_pareto.png',dpi=150); plt.show()

# ============================================================
# 6. Report Knee Solution
# ============================================================
if feas.any():
    feasible_all=np.where(pf_phys[:,1]<=0.10)[0]
    ki=feasible_all[np.argmin(pf_phys[feasible_all,0])]
    kx=X_physics[np.where((F_physics[:,0]==pf_phys[ki,0])&(F_physics[:,1]==pf_phys[ki,1]))[0][0]]
    print("\n"+"="*60)
    print("PHYSICS-VERIFIED Knee Solution")
    print("="*60)
    print(f"  xc={kx[0]:.3f}m  zc={kx[1]:.3f}m  Lx={kx[2]:.3f}m  Lz={kx[3]:.3f}m")
    print(f"  Area={pf_phys[ki,0]:.4f} m²   Physics Outage={pf_phys[ki,1]*100:.2f}%")
    print("="*60)

print("\nDone. File: two_phase_pareto.png")

# Download link
from IPython.display import FileLink,display
display(FileLink('two_phase_pareto.png'))